# Reproducing Makin et al. (2018) from dandiset 000129: bin width, training data and neurons

**Paper.** Makin JG, O'Doherty JE, Cardoso MMB, Sabes PN (2018). Superior arm-movement decoding
from cortex with a new, unsupervised-learning algorithm. *J. Neural Eng.* 15, 026010.
[doi:10.1088/1741-2552/aa9e95](https://doi.org/10.1088/1741-2552/aa9e95)

**Dataset.** [DANDI:000129](https://dandiarchive.org/dandiset/000129/0.241017.1444) version
0.241017.1444, streamed. It is the only published version; the draft holds the same two assets
and differs only in its title. The claims file, `claims/000129/10.1088-1741-2552-aa9e95.yaml`, was
written from the version of record and the paper's code (jgmakin/rbmish at commit 700062f5).

The dataset holds one session of monkey I (M1 only, 649.1 s with kinematics); the paper used 49
sessions. This notebook refits all eight decoders on that session at bin widths of 16, 32, 64 and
128 ms (figure 5), with 80, 160 and 320 s of training data (figure 6), and with fractions 1/8,
1/4, 1/2 and 1 of the units (figure 7). The data checks and the decoders at 64 ms are in
`reproduce_makin_2018.ipynb`.

| Claim | Here |
|---|---|
| fig5-advantage-with-binwidth, fig5-derivative-order | tested on one session |
| advantage-grows-with-derivative | tested on one session |
| fig7-affine-position, fig7-kinematic-model-saturates | tested on one session |
| fig5-vel-acc-all-bins, fig5-position-bins, fig6-training-time, fig7-vel-acc-neurons, fig7-position-neurons | blocked: bootstrap across the paper's 49 sessions, which the dandiset does not hold; single-session values printed |
| all others | see `reproduce_makin_2018.ipynb` |

In [ ]:
import os
os.environ.setdefault("OMP_NUM_THREADS", "4")
import time
import numpy as np
import pandas as pd
import makin as M

pd.set_option("display.width", 200)
T0 = time.time()
d = M.load_session()
print(f"dandiset {M.DANDISET} version {M.VERSION}; {d['session_description'][:80]}")

## Data check

In [ ]:
print(f"units {len(d['unit_ids'])}; kinematic samples {len(d['finger_pos'])} at {d['rate']:.0f} Hz; NaN samples {np.isnan(d['finger_pos'][:, 0]).sum()}")
rows = []
for bms in (16, 32, 64, 128):
    R, X, _ = M.bin_data(d, bms)
    ntr = int(320 / (bms / 1000))
    keep = M.unit_filter(R, bms, ntr)
    rows.append(dict(bin_ms=bms, bins=len(X), train_bins=ntr, test_bins=len(X) - ntr, nan_bins=int(np.isnan(X).any(1).sum()),
                     units_used=int(keep.sum()), mean_count=R[:, keep].mean().round(3),
                     speed_max=np.nanmax(np.hypot(X[:, 2], X[:, 3])).round(0)))
print(pd.DataFrame(rows).to_string(index=False))

The number of training bins is 320 s divided by the bin width (20,000, 10,000, 5,000 and 2,500, as
the Methods give), and the same units pass the 0.5 Hz filter at every width.

## Fits

Each configuration fits the eight decoders with seed 0 (a standard choice: one training run, as
the paper reports one per session). The rEFH is trained for max(floor(Ntrain/250), 20) epochs
(80 at 16 ms, 40 at 32 ms, 20 at 64 and 128 ms; Methods and code). For 80 and 160 s of training
the paper does not say which bins are tested; the code (getLatentsBMI.m, getTestBins) tests on
every bin after the training bins, and this is the main reading; scoring only the bins after
320 s, the same test set as for 320 s, is shown as a variant.

In [ ]:
def summary(snr):
    return pd.DataFrame({"pos": snr[["x", "y"]].mean(axis=1), "vel": snr[["vx", "vy"]].mean(axis=1),
                         "acc": snr[["ax", "ay"]].mean(axis=1)})

results, results_320, infos = {}, {}, {}
configs = [("bin", 16, dict(bin_ms=16)), ("bin", 32, dict(bin_ms=32)), ("bin", 64, dict(bin_ms=64)),
           ("bin", 128, dict(bin_ms=128)), ("train", 80, dict(train_s=80.0)), ("train", 160, dict(train_s=160.0)),
           ("frac", 1 / 8, dict(fraction=1 / 8)), ("frac", 1 / 4, dict(fraction=1 / 4)), ("frac", 1 / 2, dict(fraction=1 / 2))]
for kind, val, kw in configs:
    t = time.time()
    sp = M.prepare(d, **kw)
    info = {}
    pred = M.run_all(sp, seed=0, info=info)
    r2 = pd.DataFrame({k: sp.score(v) for k, v in pred.items()}, index=M.VARS).T.loc[M.DECODERS]
    results[(kind, val)] = r2
    infos[(kind, val)] = info
    if kind == "train":
        results_320[(kind, val)] = pd.DataFrame({k: sp.score(v, from_s=320.0) for k, v in pred.items()}, index=M.VARS).T.loc[M.DECODERS]
    print(f"{kind} {val}: units {sp.Rtr.shape[1]}, train bins {len(sp.Rtr)}, test bins {sp.okte.sum()}, "
          f"negative R2 {int((r2.values < 0).sum())}, KFlatent EM {info['kflatent']['epochs']} E steps "
          f"(last gain {info['kflatent']['last_rel_gain']:.1e}), {time.time() - t:.0f} s")
results[("train", 320)] = results[("bin", 64)]
results_320[("train", 320)] = results[("bin", 64)]
results[("frac", 1.0)] = results[("bin", 64)]
SNR = {k: summary(M.snr(v)) for k, v in results.items()}
for k, v in SNR.items():
    print(f"\n{k}, SNR (dB) averaged over x and y:"); print(v.round(2).to_string())

In [ ]:
for k, v in results.items():
    if k in (("train", 320), ("frac", 1.0)):
        continue  # the same fit as ("bin", 64)
    print(f"\n{k}, R2 per decoder and variable:"); print(v.round(3).to_string())
    print("static rEFH map, training R2:", dict(zip(M.VARS, infos[k]["refh_train_r2"].round(3))))

### Negative R2 outside 64 ms with 320 s of training

fig2-gross-failures is judged at 64 ms with all units and 320 s of training (in
`reproduce_makin_2018.ipynb`). The other configurations are listed here.

In [ ]:
neg = [(k, dec, v, round(r.loc[dec, v], 3)) for k, r in results.items() if k not in (("train", 320), ("frac", 1.0))
       for dec in r.index for v in r.columns if r.loc[dec, v] < 0]
print(pd.DataFrame(neg, columns=["config", "decoder", "variable", "R2"]).to_string(index=False))
print("decoders with a negative R2:", sorted({n[1] for n in neg}))

Outside 64 ms with all units and 320 s of training, negative R2 occurs for linear regression
(16 ms, 80 and 160 s, f = 1/8 and 1/4), static KFlatent, KFlatent (+KF) and the static rEFH
(f = 1/8 and 1/4), KFobs and the UKF (f = 1/4, x position), and KFlatent (+KF) at 16 ms
(x acceleration). Most of these are within 0.05 of zero; the gross failure is KFlatent (+KF) at
f = 1/8, whose filter diverges (R2 down to -4.8e7). R2 shown as -0.000 is negative below the
printed precision.

## fig5-advantage-with-binwidth

> Reconstruction improves with bin width, and so does the relative advantage of the rEFH (+KF).

In [ ]:
OTHERS = ["linear", "kfobs", "kflatent", "kflatent_kf", "wf", "ukf"]
widths = [16, 32, 64, 128]
def advantage(S, v):
    best = S.loc[OTHERS, v].max()
    return S.loc["refh_kf", v] - best, 100 * (S.loc["refh_kf", v] - best) / best, S.loc[OTHERS, v].idxmax()
rows = []
for w in widths:
    S = SNR[("bin", w)]
    for v in ("vel", "acc"):
        db, pc, who = advantage(S, v)
        rows.append(dict(width=w, var=v, refh_kf=S.loc["refh_kf", v], best_other=who, adv_dB=db, adv_pct=pc))
A = pd.DataFrame(rows)
print(A.round(2).to_string(index=False))
def rises(vals, tol):
    vals = np.asarray(vals)
    return bool(vals[-1] > vals[0] and np.all(np.diff(vals) >= -tol))
verdict = {}
for reading, col, tol in (("a dB", "adv_dB", 0.2), ("b percent", "adv_pct", 2.0)):
    ok = True
    for v in ("vel", "acc"):
        sub = A[A["var"] == v]
        snr_ok, adv_ok = rises(sub.refh_kf, 0.2), rises(sub[col], tol)
        print(f"reading {reading}, {v}: rEFH (+KF) SNR rises {snr_ok}; advantage rises {adv_ok}")
        ok &= snr_ok and adv_ok
    verdict[reading] = ok
print("reproduced under:", [k for k, v in verdict.items() if v])
print("SNR of every decoder by width (velocity):"); print(pd.DataFrame({w: SNR[("bin", w)]["vel"] for w in widths}).round(2).to_string())

**Result: reproduced.** The rEFH (+KF) velocity SNR rises from 3.01 dB at 16 ms to 5.46 dB at 128 ms and acceleration from 0.96 to 2.39 dB, with no fall between neighbouring widths. Its advantage over the best other decoder rises from -0.29 to 1.60 dB for velocity and from -0.44 to 1.14 dB for acceleration (-8.9 to 41.6% and -31.3 to 92.0%), so the claim holds under both readings. At 16 ms the WF beats the rEFH (+KF) for both variables. Not every decoder improves with width: the WF's velocity SNR falls from 3.30 to 3.17 dB.

## fig5-derivative-order

> With increasing derivative order reconstruction quality declines (except that the rEFH is best
> at velocity), while the relative advantage of the rEFH (+KF) increases.

In [ ]:
rows = []
for w in widths:
    S = SNR[("bin", w)]
    mean_other = S.loc[OTHERS].mean()
    pct = {v: advantage(S, v)[1] for v in ("pos", "vel", "acc")}
    q = mean_other["pos"] > mean_other["vel"] > mean_other["acc"]
    a = pct["pos"] < pct["vel"] < pct["acc"]
    rows.append(dict(width=w, mean6_pos=mean_other["pos"], mean6_vel=mean_other["vel"], mean6_acc=mean_other["acc"],
                     adv_pct_pos=pct["pos"], adv_pct_vel=pct["vel"], adv_pct_acc=pct["acc"], quality_declines=q,
                     advantage_rises=a, both=q and a, refh_kf_best_var=S.loc["refh_kf"].idxmax()))
D = pd.DataFrame(rows)
print(D.round(2).to_string(index=False))
print("widths where both hold:", int(D.both.sum()), "of 4")

**Result: partially reproduced.** The mean SNR of the six other decoders falls from position to velocity to acceleration at all four widths (at 64 ms 3.79, 3.06 and 0.99 dB). The percent advantage of the rEFH (+KF) rises from position to velocity to acceleration at 32, 64 and 128 ms (at 64 ms 4.9, 32.6 and 76.5%), but not at 16 ms, where it is negative and falls (-6.2, -8.9 and -31.3%). Both parts hold at 3 of 4 widths. The claim's exception does not hold here: the rEFH (+KF) decodes position best at every width.

## advantage-grows-with-derivative

> At 64 ms and with each amount of training data, the relative advantage of the rEFH (+KF) over
> the other decoders increases from position to velocity to acceleration.

In [ ]:
for label, res in (("test on all bins after training (main)", results), ("test on bins after 320 s (variant)", results_320)):
    rows = []
    for tr in (80, 160, 320):
        S = summary(M.snr(res[("train", tr)]))
        pct = {v: advantage(S, v)[1] for v in ("pos", "vel", "acc")}
        rows.append(dict(train_s=tr, adv_pct_pos=pct["pos"], adv_pct_vel=pct["vel"], adv_pct_acc=pct["acc"],
                         best_other_pos=advantage(S, "pos")[2], rises=pct["pos"] < pct["vel"] < pct["acc"]))
    print(label); print(pd.DataFrame(rows).round(2).to_string(index=False))
for tr in (80, 160):
    print(f"\nSNR (dB), {tr} s of training, main test set:"); print(summary(M.snr(results[('train', tr)])).round(2).to_string())

**Result: reproduced.** The percent advantage of the rEFH (+KF) rises from position to velocity to acceleration with 80 s (-26.1, 10.9 and 76.4%), 160 s (2.4, 32.5 and 71.3%) and 320 s (4.9, 32.6 and 76.5%) of training. It also rises at each amount when the shorter trainings are scored on the bins after 320 s only.

## fig7-affine-position

> Position SNR of the decoders without a learned kinematic model (linear regression, static
> KFlatent, WF, static rEFH) is close to an affine function of the fraction of neurons, with
> correlations 0.999, 0.95, 0.98 and 1.000; static KFlatent is the most curved.

In [ ]:
fr = [1 / 8, 1 / 4, 1 / 2, 1.0]
P = pd.DataFrame({f: SNR[("frac", f)]["pos"] for f in fr})
P.columns = ["1/8", "1/4", "1/2", "1"]
print("position SNR (dB), averaged over x and y:"); print(P.round(2).to_string())
static = ["linear", "kflatent", "wf", "refh"]
expected = dict(linear=0.999, kflatent=0.95, wf=0.98, refh=1.0)
out = {}
for reading, xs in (("a f", np.array(fr)), ("b log2 f", np.log2(fr))):
    rho = {k: np.corrcoef(xs, P.loc[k].values)[0, 1] for k in static}
    within = {k: abs(rho[k] - expected[k]) <= 0.1 * expected[k] for k in static}
    lowest = min(rho, key=rho.get)
    out[reading] = dict(**{f"rho_{k}": rho[k] for k in static}, all_within=all(within.values()), lowest=lowest,
                        holds=all(within.values()) and lowest == "kflatent")
print(pd.DataFrame(out).T.to_string())

**Result: not reproduced.** The correlations of position SNR with f are 0.995 (linear regression), 0.995 (static KFlatent), 0.986 (WF) and 0.991 (static rEFH), and with log2 f 0.956, 0.948, 0.954 and 0.914; all are within 10% of the paper's values. But static KFlatent does not have the lowest correlation under either reading (the WF has with f, the static rEFH with log2 f), so the ordering condition fails. At f = 1/8 and 1/4 (13 and 27 units) the position SNR of these four decoders is within 0.33 dB of zero.

## fig7-kinematic-model-saturates

> Position SNR of the four decoders that learn a kinematic state-transition model (KFobs,
> KFlatent (+KF), UKF, rEFH (+KF)) rises much faster at first with added neurons, then saturates.

In [ ]:
dyn = ["kfobs", "kflatent_kf", "ukf", "refh_kf"]
Pf = pd.DataFrame({f: SNR[("frac", f)]["pos"] for f in fr})
print(Pf.round(2).to_string())
late = (Pf[1.0] - Pf[0.5]) / 0.5
res = {}
for reading, (f0, f1) in (("a 1/8 to 1/4", (1 / 8, 1 / 4)), ("b 1/8 to 1/2", (1 / 8, 1 / 2))):
    init = (Pf[f1] - Pf[f0]) / (f1 - f0)
    tab = pd.DataFrame({"initial_slope": init, "late_slope": late})
    max_static_init = init[static].max()
    ok = {k: bool(init[k] > late[k] and init[k] > max_static_init) for k in dyn}
    print(f"\nreading {reading} (dB per unit f):"); print(tab.round(2).to_string())
    print(f"largest initial slope among the static decoders: {max_static_init:.2f}; per dynamic decoder: {ok}")
    res[reading] = all(ok.values())
print("\nreproduced under:", [k for k, v in res.items() if v])

**Result: reproduced, under reading b.** From f = 1/8 to 1/2 the position SNR of KFobs, KFlatent (+KF), the UKF and the rEFH (+KF) rises 6.67, 207.33, 6.93 and 6.22 dB per unit f. That is faster than from 1/2 to 1 (3.79, 6.27, 4.95 and 5.79) and faster than any static decoder's initial rise (largest 5.00, static KFlatent). Under reading a (1/8 to 1/4) only KFlatent (+KF) passes: KFobs, the UKF and the rEFH (+KF) decode position slightly worse at 1/4 than at 1/8. KFlatent (+KF)'s large initial slope comes from a filter that diverges at f = 1/8 (position SNR -76.63 dB), a gross failure.

## Single-session parts of claims blocked for the other sessions

The claims below need a bootstrap over the paper's 49 sessions, which the dandiset does not hold,
so they stay blocked. Their point comparisons on this session are printed here; they do not decide
an outcome.

### fig5-vel-acc-all-bins

> For velocity and acceleration, the rEFH (+KF) is the best decoder at every bin width
> (16, 32, 64 and 128 ms).

In [ ]:
rows = []
for w in widths:
    S = SNR[("bin", w)]
    for v in ("vel", "acc"):
        rest = S.drop(index="refh_kf")[v]
        rows.append(dict(width=w, var=v, refh_kf=S.loc["refh_kf", v], next_best=rest.idxmax(),
                         margin=S.loc["refh_kf", v] - rest.max()))
print(pd.DataFrame(rows).round(2).to_string(index=False))

### fig5-position-bins

> For position the leading decoders cannot be told apart at any bin width except 16 ms, where the
> UKF is better.

In [ ]:
rows = []
for w in widths:
    S = SNR[("bin", w)]["pos"]
    rows.append(dict(width=w, refh_kf=S["refh_kf"], ukf=S["ukf"], kfobs=S["kfobs"], ukf_minus_refh_kf=S["ukf"] - S["refh_kf"],
                     kfobs_minus_refh_kf=S["kfobs"] - S["refh_kf"], ukf_minus_kfobs=S["ukf"] - S["kfobs"]))
print("position SNR (dB):"); print(pd.DataFrame(rows).round(2).to_string(index=False))

### fig6-training-time

> The rEFH (+KF) is the best decoder of velocity and acceleration for 80, 160 and 320 s of training
> data; for position it never differs significantly from the UKF, and beats KFobs at 320 s but loses
> to it at 80 s.

In [ ]:
for label, res in (("test on all bins after training (main)", results), ("test on bins after 320 s (variant)", results_320)):
    rows = []
    for tr in (80, 160, 320):
        S = summary(M.snr(res[("train", tr)]))
        row = dict(train_s=tr)
        for v in ("vel", "acc"):
            rest = S.drop(index="refh_kf")[v]
            row[f"{v}_best"] = S[v].idxmax(); row[f"{v}_margin"] = S.loc["refh_kf", v] - rest.max()
        row.update(pos_refh_kf=S.loc["refh_kf", "pos"], pos_ukf=S.loc["ukf", "pos"], pos_kfobs=S.loc["kfobs", "pos"],
                   pos_minus_ukf=S.loc["refh_kf", "pos"] - S.loc["ukf", "pos"], pos_minus_kfobs=S.loc["refh_kf", "pos"] - S.loc["kfobs", "pos"])
        rows.append(row)
    print(label); print(pd.DataFrame(rows).round(2).to_string(index=False))

### fig7-vel-acc-neurons

> For velocity and acceleration, using a fraction 1/4, 1/2 or all of the neurons, the rEFH (+KF)
> outperforms all its rivals, and its percent gain in SNR grows with the number of neurons and with
> derivative order.

Percent gain under reading (a), over the best decoder other than the two rEFH decoders, and
reading (b), over each such decoder.

In [ ]:
rows, pct_each = [], {}
for f in (1 / 4, 1 / 2, 1.0):
    S = SNR[("frac", f)]
    for v in ("vel", "acc"):
        rest = S.drop(index="refh_kf")[v]
        best = S.loc[OTHERS, v].max()
        rows.append(dict(f=f, var=v, refh_kf=S.loc["refh_kf", v], next_best=rest.idxmax(), margin=S.loc["refh_kf", v] - rest.max(),
                         pct_over_best_other=100 * (S.loc["refh_kf", v] - best) / best))
        pct_each[(f, v)] = 100 * (S.loc["refh_kf", v] - S.loc[OTHERS, v]) / S.loc[OTHERS, v]
print(pd.DataFrame(rows).round(2).to_string(index=False))
print("\npercent gain over each decoder (reading b):"); print(pd.DataFrame(pct_each).round(1).to_string())

### fig7-position-neurons

> For position the rEFH's advantage is significant at p < 0.05 at f = 1/4 and 1/2, but not at
> f = 1, where the UKF comes close, nor at f = 1/8, where all decoders collapse.

In [ ]:
rows = []
for f in fr:
    S = SNR[("frac", f)]["pos"]
    rest = S.drop("refh_kf")
    rows.append(dict(f=f, refh_kf=S["refh_kf"], next_best=rest.idxmax(), margin=S["refh_kf"] - rest.max(),
                     minus_ukf=S["refh_kf"] - S["ukf"]))
print("position SNR (dB):"); print(pd.DataFrame(rows).round(2).to_string(index=False))

**Result: blocked, single-session values only.** These claims need the bootstrap over the paper's
49 sessions. On this session:
- fig5-vel-acc-all-bins: the rEFH (+KF) has the highest velocity and acceleration SNR at 32, 64 and
  128 ms (margins 0.53 and 0.32, 1.19 and 0.45, 0.45 and 0.12 dB), but at 16 ms the WF is higher
  for both (3.30 against 3.01 dB, and 1.39 against 0.96 dB).
- fig5-position-bins: at 16 ms the UKF's position SNR exceeds the rEFH (+KF)'s by 0.29 dB, the
  direction the paper reports. At 32, 64 and 128 ms the differences between the rEFH (+KF), the
  UKF and KFobs are 0.01 to 1.22 dB in size; whether any is significant needs the bootstrap.
- fig6-training-time: part (i) holds on point values, the rEFH (+KF) having the highest velocity
  and acceleration SNR at 80, 160 and 320 s (velocity margins 0.33, 1.01 and 1.19 dB, acceleration
  0.68, 0.47 and 0.45 dB). For position it is 0.87 dB below the UKF at 80 s and 0.10 and 0.28 dB
  above it at 160 and 320 s, and against KFobs it is above at 320 s (0.90 dB) and below at 80 s
  (0.20 dB), the directions part (iii) states. Scoring on the bins after 320 s gives the same
  signs.
- fig7-vel-acc-neurons: at f = 1/4 the rEFH (+KF) ties the WF for velocity (1.56 dB each, 0.00 dB
  apart) and exceeds it by 0.20 dB for acceleration; at f = 1/2 and 1 it is the best decoder of
  both. Its percent gain over the best decoder other than the two rEFH decoders (reading a) rises
  with f for velocity (0.2, 19.1 and 32.6%) but not for acceleration (38.2, 87.1 and 76.5%); it is
  larger for acceleration than velocity at each f. Over each decoder (reading b) the gain also falls
  somewhere, for example over the UKF for velocity from 19.4% at f = 1/4 to 19.1% at f = 1/2.
- fig7-position-neurons: the rEFH (+KF) has the highest position SNR at f = 1/8 (by 0.07 dB; every
  decoder but the diverged KFlatent (+KF) within 0.7 dB of zero), at f = 1/4 (by 0.21 dB) and at f = 1 (0.28 dB over the UKF),
  and is 0.15 dB below the UKF at f = 1/2, against the paper's advantage there.

## Summary

| Claim | Outcome |
|---|---|
| fig5-advantage-with-binwidth | reproduced |
| fig5-derivative-order | partially reproduced (3 of 4 widths; not at 16 ms) |
| advantage-grows-with-derivative | reproduced |
| fig7-affine-position | not reproduced (correlations within tolerance; static KFlatent not the most curved) |
| fig7-kinematic-model-saturates | reproduced under reading b |
| fig5-vel-acc-all-bins, fig5-position-bins, fig6-training-time, fig7-vel-acc-neurons, fig7-position-neurons | blocked: bootstrap across the paper's 49 sessions, which the dandiset does not hold (single-session values above) |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the
reasoning and the differences from the paper's method.

**How the analysis differs from the paper's.** One session (dated 2017-02-02, not in the paper's
session list) in place of 49, so each mean over sessions is this session's value. Sorted units
only, 111 above 0.5 Hz, without hash units; a fraction f keeps the first floor(111 f) of them
(13, 27 and 55 units). For 80 and 160 s of training the test bins are all bins after the
training bins, as in the paper's code; the paper does not say. The decoders are the same
re-implementations as in `reproduce_makin_2018.ipynb`; KFlatent's EM reached the 100-epoch limit
in every configuration without meeting the 1/100 of a percent criterion (printed with the fits).

**What was found about the dataset.** As in `reproduce_makin_2018.ipynb`: one session, no hash
units, three undocumented 0.1 s gaps, and a test file without kinematics. The same 111 units pass
the rate filter at every bin width.

In [ ]:
print(f"notebook run time {(time.time() - T0) / 60:.1f} min")